# Load data

In [1]:
import pandas as pd

In [2]:
selected_data = pd.read_pickle(r"..\..\..\data\preprocessed\full_data_organizations.pkl")

# Hugging Face (*Transformers*) 

In [3]:
import torch

from transformers import pipeline, BertTokenizer

Import model

In [4]:
distilled_student_sentiment_classifier = pipeline(
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student", 
    return_all_scores=True
)

c:\Users\eduardo.moreno\AppData\Local\anaconda3\envs\env_nlp\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Import tokenizer

In [5]:
tokenizer = BertTokenizer.from_pretrained('lxyuan/distilbert-base-multilingual-cased-sentiments-student')

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


Example of use and response of the model

In [6]:
distilled_student_sentiment_classifier("I love this movie and i would watch it again and again!")

[[{'label': 'positive', 'score': 0.9731044769287109},
  {'label': 'neutral', 'score': 0.016910076141357422},
  {'label': 'negative', 'score': 0.009985473938286304}]]

Let's use it with our data

In [7]:
print(selected_data.iloc[418233]['Message'])

Al celebrarse hoy, 5 de junio, el Día Mundial del Medio Ambiente desde el 1954 el Ministerio de Defensa, sus instituciones y dependencias, reafirma su apoyo incondicional a la conservación del medio ambiente en toda su área de responsabilidad en el territorio de la República Dominicana, sus espacios terrestres, marítimos y aéreos. El MIDE se une hoy al Ministerio de Medio Ambiente y Recursos Naturales, razón por la cual seguimos realizando operaciones en todo el territorio nacional para proteger todos los ecosistemas dominicanos. Luchamos contra la tala de árboles para conservar las cuencas acuíferas e hidrográficas que dan origen a los ríos y arroyos que proveen al pueblo dominicano del preciado liquido, como es agua potable. De igualmente manera, ante los retos que nos impone el cambio climático, dentro de nuestra planificación estratégica está concebida la conservación de energía así como el fomento para el uso de fuentes alternativas como la energía solar. El Día Mundial del Medio 

In [8]:
distilled_student_sentiment_classifier(selected_data.iloc[418233]['Message'])

[[{'label': 'positive', 'score': 0.6169759035110474},
  {'label': 'neutral', 'score': 0.07673612982034683},
  {'label': 'negative', 'score': 0.3062879145145416}]]

`distilbert-base-multilingual-cased-sentiments-student` as it base model `bert` accepts only $512$ tokens, so what do you think will happen if we try to analyze an input **greater than 512 tokens**?


In [9]:
txt = selected_data['Message'][0]
print(txt)

A LOS NACIDOS ENTRE 1970 Y 1987 Somos una generación especial y nos denominaron la generación X y como no, si nuestra infancia estuvo llena de cambios Somos la última generación que jugaba en la calle y en los recreos del colegio a las bolitas, a el escondite, somos la primera generación que jugó con videojuegos, fuimos a parques de atracciones y vimos caricaturas a color. Fuimos los últimos en grabar canciones de la radio en casettes (como olvidarlo si mientras se grababa en algunos casos no podíamos ni hablar) y vimos películas en versión Beta y VHS PERO orgullosos pioneros del personal stereo y los CD's. Cuantos no tuvimos que tragarnos, Salvado por la Campana (con todo y Screech), Beverly Hills 90210. Nosotros vimos la caída de torres gemelas y también vimos caer el muro de Berlín. Aprendimos a utilizar los ordenadores antes que nuestros padres y abuelos, y sobre todo antes de todos esos niños cerebritos de hoy en día y nunca vimos a los que no sabían usar los ordenadores como una 

In [10]:
tokens = tokenizer.encode_plus(txt, add_special_tokens=False)

len(tokens['input_ids'])

Token indices sequence length is longer than the specified maximum sequence length for this model (1201 > 512). Running this sequence through the model will result in indexing errors


1201

In [11]:
tokens = tokenizer.encode_plus(txt, add_special_tokens=False,
                               return_tensors='pt')

print(len(tokens['input_ids'][0]))
tokens

1201


{'input_ids': tensor([[  138,   149, 21793,  ..., 12816, 12882, 10466]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}

Here we have specified a few arguments that require some explanation.

* `max_length` - this tell the tokenizer the maximum number of tokens we want to see in each sample, for BERT we almost always use `512` as that is the length of sequences that BERT consumes.

* `truncation` - if our input string `txt` contains more tokens than allowed (specified in `max_length` parameter) then we cut all tokens past the `max_length` limit.

* `padding` - if our input string `txt` contains less tokens than specified by `max_length` then we pad the sequence with zeros (`0` is the token ID for *'[PAD]'* - BERTs padding token).

* `add_special_tokens` - whether or not to add special tokens, when using BERT we always want this to be `True` unless we are adding them ourselves.

| Token | ID | Description |
| --- | --- | --- |
| [PAD] | 0 | Used to fill empty space when input sequence is shorter than required sequence size for model |
| [UNK] | 100 | If a word/character is not found in BERTs vocabulary it will be represented by this *unknown* token |
| [CLS] | 101 | Represents the start of a sequence |
| [SEP] | 102 | Seperator token to denote the end of a sequence and as a seperator where there are multiple sequences |
| [MASK] | 103 | Token used for masking other tokens, used for masked language modeling |

*Note that our tokenized sequence begins with `101`, the seperator token `102` can be found seperating the input sequence and padding tokens `0`.*

* `return_tensors` - here we specify either `'pt'` to return PyTorch tensors, or `'tf'` to return TensorFlow tensors.

In [12]:
try:
    response = distilled_student_sentiment_classifier(txt)
    print(response)
except Exception as e:
    print(f"Something went WRONG: {e}")

Token indices sequence length is longer than the specified maximum sequence length for this model (1203 > 512). Running this sequence through the model will result in indexing errors


Something went WRONG: The size of tensor a (1203) must match the size of tensor b (512) at non-singleton dimension 1


What happen if we only send a chunk of the post?

In [13]:
try:
    response = distilled_student_sentiment_classifier(txt[:1500])
    print(response)
except Exception as e:
    print(f"Something went WRONG: {e}")

[[{'label': 'positive', 'score': 0.28041714429855347}, {'label': 'neutral', 'score': 0.1233644187450409}, {'label': 'negative', 'score': 0.5962184071540833}]]


We will use [LangChain](https://python.langchain.com/v0.2/docs/introduction/) to split posts so we can compute the sentiment of each split.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1100,  # Ajusta según sea necesario
    chunk_overlap=200,
    separators=[" "]
)

In [16]:
texts = text_splitter.create_documents([txt])
print(len(texts))

5


In [17]:
len(texts[0].page_content)

1099

In [18]:
texts[0].page_content[-110:]

's como una especie de "retardados" como sucede hoy. Jugamos con el Double Dragon, Street Fighter, empezamos el'

In [19]:
texts[1].page_content[100:]

'especie de "retardados" como sucede hoy. Jugamos con el Double Dragon, Street Fighter, empezamos el Mortal Kombat, el tetris, el Mario Bross, nos pegábamos a la tele a mirar Hugo y como otros jugaban desde sus casas usando el teléfono de red fija!! vimos los anuncios de los primeros celulares (que parecían ladrillos) y creímos que Internet sería un mundo libre. Somos la Generación de Xuxa, Robotech, Gi Joe, Los Halcones Galácticos, los ThunderCats, los Transformers, He-Man y las Tortugas Ninja, Del Correcaminos, Los Supercampeones, Espartaco, Mazinger, de los Pitufos, La Pantera Rosa, Los Picapiedras, El pájaro loco, Candi Candi, Remi y Marco. Los que crecieron escuchando a Madonna, Michael Jackson y Guns N\'Roses, New Kids on the block, Por supuesto en ver y vivir los primeros VIDEOS MUSICALES , Los Locomía y sus abanicos. La última generación de las botellas de litro de Coca-Cola familiar cuando un litro alcanzaba para toda la familia!! Y los últimos en ser mandados a comprar en la'

In [20]:
pos_sent = []
neu_sent = []
neg_sent = []

for text_i in texts:
    response = distilled_student_sentiment_classifier(text_i.page_content)

    print(text_i.page_content)
    print(response, '\n')

    for sent in response[0]:
        if sent['label'] == 'positive':
            pos_sent.append(sent['score'])
        elif sent['label'] == 'neutral':
            neu_sent.append(sent['score'])
        elif sent['label'] == 'negative':
            neg_sent.append(sent['score'])

A LOS NACIDOS ENTRE 1970 Y 1987 Somos una generación especial y nos denominaron la generación X y como no, si nuestra infancia estuvo llena de cambios Somos la última generación que jugaba en la calle y en los recreos del colegio a las bolitas, a el escondite, somos la primera generación que jugó con videojuegos, fuimos a parques de atracciones y vimos caricaturas a color. Fuimos los últimos en grabar canciones de la radio en casettes (como olvidarlo si mientras se grababa en algunos casos no podíamos ni hablar) y vimos películas en versión Beta y VHS PERO orgullosos pioneros del personal stereo y los CD's. Cuantos no tuvimos que tragarnos, Salvado por la Campana (con todo y Screech), Beverly Hills 90210. Nosotros vimos la caída de torres gemelas y también vimos caer el muro de Berlín. Aprendimos a utilizar los ordenadores antes que nuestros padres y abuelos, y sobre todo antes de todos esos niños cerebritos de hoy en día y nunca vimos a los que no sabían usar los ordenadores como una 

For the final output we will compute the average of sentiment on each partition and will return the highest as the final sentiment of the initial post.

In [21]:
import numpy  as np

In [22]:
mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]

In [23]:
mean_scores

[0.38365131467580793, 0.13874634131789207, 0.47760233134031294]

In [24]:
sentiment = np.argmax(mean_scores)
score = np.max(mean_scores)

if sentiment == 0:
    print(f"The sentiment of the post is POSITIVE: {score}")
if sentiment == 1:
    print(f"The sentiment of the post is NEUTRAL: {score}")
if sentiment == 2:
    print(f"The sentiment of the post is NEGATIVE: {score}")

The sentiment of the post is NEGATIVE: 0.47760233134031294


Now let's compute the sentiment for each post!

In [25]:
from tqdm import tqdm

In [26]:
def get_sentiment(post):
    try:
        response = distilled_student_sentiment_classifier(post)
        pos_sent = []
        neu_sent = []
        neg_sent = []
        for sent in response[0]:
            if sent['label'] == 'positive':
                pos_sent.append(sent['score'])
            elif sent['label'] == 'neutral':
                neu_sent.append(sent['score'])
            elif sent['label'] == 'negative':
                neg_sent.append(sent['score'])
        mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]
        sentiment = np.argmax(mean_scores)
        score = np.max(mean_scores)
        if sentiment == 0:
            sentiment = 'Positive'
        elif sentiment == 1:
            sentiment = 'Neutral'
        else:
            sentiment = 'Negative'

    except Exception as e:
        # Split the post
        texts = text_splitter.create_documents([post])
        # Compute the sentiment for each split
        pos_sent = []
        neu_sent = []
        neg_sent = []
        for text_i in texts:
            response = distilled_student_sentiment_classifier(text_i.page_content)
            # Compute avg sentiment
            for sent in response[0]:
                if sent['label'] == 'positive':
                    pos_sent.append(sent['score'])
                elif sent['label'] == 'neutral':
                    neu_sent.append(sent['score'])
                elif sent['label'] == 'negative':
                    neg_sent.append(sent['score'])
        # Compute final sentiment and score
        mean_scores = [np.mean(pos_sent), np.mean(neu_sent), np.mean(neg_sent)]
        sentiment = np.argmax(mean_scores)
        score = np.max(mean_scores)
        if sentiment == 0:
            sentiment = 'Positive'
        elif sentiment == 1:
            sentiment = 'Neutral'
        else:
            sentiment = 'Negative'
    
    return {'sentiment': sentiment, 'score': score}


In [27]:
get_sentiment(txt)

{'sentiment': 'Negative', 'score': 0.47760233134031294}

In [28]:
df_aux = pd.DataFrame()

list_msgs = selected_data['Message'].to_list()[:10]

In [29]:
list_sentiment = []
list_scores = []

for post_i in tqdm(list_msgs):
    response = get_sentiment(post_i)
    list_sentiment.append(response['sentiment'])
    list_scores.append(response['score'])

100%|██████████| 10/10 [00:07<00:00,  1.29it/s]


# References

- [Hugging Face Bert](https://huggingface.co/docs/transformers/model_doc/bert)
- [Intro to Tokenizer for Bert](https://medium.com/@dhartidhami/understanding-bert-word-embeddings-7dc4d2ea54ca)
- [Hugging Face model DistilBert](https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student?) 